# Drone Görüntülerinde Araç Tespiti · YOLO11s Baseline

**LEVEL-UP AI | ROKETSAN Yapay Zekâ Hackathonu · Aşama 1**

| | |
|---|---|
| **Görev** | Drone görüntülerinde alanı ≥ 200 px² olan araçları tespit edip `car`, `van`, `truck`, `bus` olarak sınıflandırmak |
| **Metrik** | mAP@0.5 (dört sınıfın AP ortalaması, her sınıf eşit ağırlıklı) |
| **Model** | YOLO11s (Ultralytics), COCO ön-eğitimli |
| **Doğrulama** | `splits/scene_holdout_v2`: sahne grupları bölünmeden, test dağılımına göre katmanlanmış 1.295 görüntü |
| **Donanım** | Kaggle, 2× NVIDIA T4 (DDP) |

Bu notebook baştan sona **Run All** ile çalışır: veriyi hazırlar, modeli eğitir, yarışma metriğiyle değerlendirir ve `submission.csv` üretir.

### İçindekiler
1. [Ortam ve yapılandırma](#1)
2. [Veri seti](#2)
3. [Sınıf dengesizliği: Repeat Factor Sampling](#3)
4. [LR'ye bağlı azalan veri artırımı](#4)
5. [Eğitim](#5)
6. [Eğitim geçmişi](#6)
7. [Değerlendirme](#7)
8. [Test tahmini ve submission](#8)
9. [Özet](#9)

## Tasarım kararları ve dayanakları

Kararlar keşifsel veri analizine (`hakan/notebooks/01_eda.ipynb`, `furkan/notebooks/01b_eda_addendum.ipynb`) dayanır.
Doğrulanmamış varsayımlara dayanan ayarlar eklenmemiş, bunlar için Ultralytics varsayılanları korunmuştur.

| Karar | Değer | Dayanak |
|---|---|---|
| Giriş çözünürlüğü | `imgsz=1280` | Kısa kenarı 16 px'in altına düşen kutular: 960'ta %38, 1280'de %20 |
| Dikey çevirme | `flipud=0` | Eğik çekim perspektifi: nesne boyutu görüntünün üstünden altına 2,4 kat artıyor (Spearman ρ = 0,38) |
| Yatay çevirme | `fliplr=0.5` | Sahnede sol-sağ simetrisi korunuyor |
| Sınıf dengesizliği | Repeat Factor Sampling | Kutu payları car %75 · van %14 · truck %7 · bus %3; mAP'te her sınıf %25 ağırlıklı |
| Veri artırımı takvimi | LR ile azalan, son 10 epoch mosaic kapalı | Eğitim sonunda gerçek görüntü dağılımına yaklaşmak (`close_mosaic` fikrinin sürekli hâli) |
| Tahmin eşikleri | `conf=0.001`, `max_det=1000` | Yarışma kuralı: düşük güvenli kutular cezalandırılmaz; görüntü başına en fazla 218 kutu |
| Doğrulama ağırlıkları | Çözünürlük × aydınlatma katmanları | Test dağılımı train'den farklı (1400×788 testin %44'ü, bunların %45'i karanlık) |
| Early stopping | Kapalı | Süre bütçeli cosine takvimin düşük LR evresi korunur; `last.pt` ve `best.pt` sonda karşılaştırılır |

<a id="1"></a>
## 1 · Ortam ve yapılandırma

Tüm ayarlar tek yerde. `TIME_H` eğitim bütçesidir (doğrulama dahil); Ultralytics epoch sayısını ve öğrenme oranı takvimini bu süreye göre ayarlar.

In [ ]:
import os
import sys
import json
import math
import time
import random
import shutil
import zipfile
import subprocess
from pathlib import Path

T0 = time.time()

# --- Deney ayarları -----------------------------------------------------------------------------
TIME_H    = float(os.environ.get("TIME_H", 2.5))    # eğitim bütçesi (saat)
IMGSZ     = int(os.environ.get("IMGSZ", 1280))
BATCH     = int(os.environ.get("BATCH", 16))        # iki GPU toplamı; bellek yetmezse Ultralytics yarıya indirir
EPOCHS    = int(os.environ.get("EPOCHS", 300))      # üst sınır; gerçek sayıyı TIME_H belirler
MODEL     = os.environ.get("MODEL", "yolo11s.pt")
DEVICE    = os.environ.get("DEVICE", "0,1")
RFS_T     = 0.5                                      # Repeat Factor Sampling eşiği
AUG_FLOOR = 0.3                                      # veri artırımının ineceği en düşük oran
CLOSE_N   = int(os.environ.get("CLOSE_N", 10))      # mosaic'in kapalı olduğu son epoch sayısı
DARK_THR  = 55                                       # karanlık görüntü eşiği (64×64 gri ortalama; v1/v2 ile aynı)
SEED      = 42
CLASSES   = ["car", "van", "truck", "bus"]
ULTRALYTICS_VERSION = "8.4.163"                      # kodun test edildiği sürüm

# --- Yollar -------------------------------------------------------------------------------------
SMOKE = os.environ.get("SMOKE") == "1"               # yalnızca CPU üzerinde akış testi için
INPUT = Path(os.environ.get("INPUT_DIR", "/kaggle/input")).resolve()
WORK  = Path(os.environ.get("WORK_DIR", "/kaggle/working")).resolve()
DS    = WORK / "yolo_v2"
RUN_NAME = "y11s_v2"


def log(*args):
    print(f"[{(time.time() - T0) / 60:6.1f} dk]", *args, flush=True)

In [ ]:
import torch

TORCH_VERSION = torch.__version__
log(f"torch {TORCH_VERSION} | CUDA: {torch.cuda.is_available()} | GPU sayısı: {torch.cuda.device_count()}")
if not SMOKE:
    subprocess.run(["nvidia-smi", "-L"], check=False)
    assert torch.cuda.device_count() >= 2, "2× T4 bekleniyor: Settings → Accelerator → GPU T4 x2"

try:
    import ultralytics
    assert ultralytics.__version__ == ULTRALYTICS_VERSION
except (ImportError, AssertionError):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)
    import importlib, ultralytics
    importlib.reload(ultralytics)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

assert torch.__version__ == TORCH_VERSION, "pip torch sürümünü değiştirdi"
pd.set_option("display.float_format", "{:,.4f}".format)
log("ultralytics", ultralytics.__version__)

<a id="2"></a>
## 2 · Veri seti

Kaggle dataset'i (`roketsan-yolo-v2`) yedi zip parçasından oluşur. Kaggle her parçayı ayrı bir klasöre açtığı için
tüm `yolo_v2/` dosyaları tek bir çalışma klasöründe birleştirilir. Etiketler YOLO formatındadır
(`sınıf cx cy w h`, normalize); sınıf sırası `car=0, van=1, truck=2, bus=3`.

In [ ]:
LIST_FILES = {"train.txt", "val.txt", "test.txt", "val_1400x788.txt", "val_dark.txt",
              "val_1400x788_dark.txt", "classes.json"}


def dataset_relpath(p: Path):
    """Bir dosyanın yolo_v2/ içindeki göreli yolunu bulur; Kaggle'ın zip açma biçiminden bağımsızdır."""
    parts = p.parts
    if "yolo_v2" in parts:                                   # .../yolo_v2/images/train/x.jpg
        return Path(*parts[len(parts) - 1 - parts[::-1].index("yolo_v2") + 1:])
    for k in range(len(parts) - 2, 0, -1):                   # .../images/train/x.jpg (üst klasör atılmışsa)
        if parts[k - 1] in ("images", "labels") and parts[k] in ("train", "test"):
            return Path(*parts[k - 1:])
    if p.name in LIST_FILES:                                 # liste dosyaları kök dizinde kalmışsa
        return Path(p.name)
    return None


def walk_files(root: Path):
    """Tüm dosyaları gezer; Kaggle input klasörleri symlink olabileceği için linkleri takip eder."""
    for dirpath, _, filenames in os.walk(root, followlinks=True):
        for f in filenames:
            yield Path(dirpath) / f


def assemble_dataset():
    """/kaggle/input altındaki veri seti parçalarını DS altında birleştirir (açılmamış zip'ler dahil)."""
    if (DS / "classes.json").exists():
        return 0
    unz = WORK / "_unz"
    for z in (q for q in walk_files(INPUT) if q.suffix == ".zip"):
        with zipfile.ZipFile(z) as f:
            f.extractall(unz)
    n = 0
    for root in (INPUT, unz):
        if not root.exists():
            continue
        for p in walk_files(root):
            rel = dataset_relpath(p)
            if rel is not None:
                dst = DS / rel
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copyfile(p, dst)
                    n += 1
    shutil.rmtree(unz, ignore_errors=True)
    if not (DS / "classes.json").exists():
        tree = sorted(str(q.relative_to(INPUT)) for q in walk_files(INPUT))[:40] if INPUT.exists() else []
        raise FileNotFoundError(
            "Veri seti bulunamadı. Sağ panelden 'Add Input' → Your Datasets → roketsan-yolo-v2 ekli mi ve "
            "yüklemesi tamamlandı mı kontrol edin.\n/kaggle/input içeriği (ilk 40):\n  " + "\n  ".join(tree or ["<boş>"]))
    return n


log(f"{assemble_dataset()} dosya birleştirildi → {DS}")
counts = {k: len(list((DS / k).glob("*"))) for k in ["images/train", "labels/train", "images/test"]}
if not SMOKE:
    assert counts == {"images/train": 6471, "labels/train": 6471, "images/test": 2118}, counts


def read_ids(name):
    return [Path(s).stem for s in (DS / name).read_text().split() if s.strip()]


train_ids, val_ids, test_ids = read_ids("train.txt"), read_ids("val.txt"), read_ids("test.txt")
assert not set(train_ids) & set(val_ids), "train ve val kesişiyor"
pd.DataFrame({"görüntü": [len(train_ids), len(val_ids), len(test_ids)]}, index=["train", "val (v2)", "test"])

In [ ]:
# Etiket istatistikleri (train + val)
rows = []
for split, ids in [("train", train_ids), ("val", val_ids)]:
    for i in ids:
        for s in (DS / "labels/train" / f"{i}.txt").read_text().splitlines():
            if s.strip():
                rows.append((split, CLASSES[int(s.split()[0])]))
lab = pd.DataFrame(rows, columns=["split", "label"])
dist = pd.crosstab(lab.label, lab.split).reindex(CLASSES)
dist_pct = (dist / dist.sum() * 100).round(2).add_suffix(" %")
pd.concat([dist, dist_pct], axis=1)

<a id="3"></a>
## 3 · Sınıf dengesizliği: Repeat Factor Sampling

Kutu bazında `car` %75, `bus` ise yalnızca %3'tür; ama mAP'te her sınıf eşit ağırlıklıdır.
Kayıp fonksiyonunu değiştirmek yerine, uzun kuyruklu tespit için önerilen **Repeat Factor Sampling** (LVIS, Gupta ve ark., 2019) uygulanır:

$$ r_c = \max\left(1, \sqrt{t / f_c}\right), \qquad r_{\text{görüntü}} = \max_{c \,\in\, \text{görüntü}} r_c $$

$f_c$: sınıfı içeren train görüntülerinin oranı, $t = 0.5$. Kesirli tekrar stokastik yuvarlanır (seed 42).
Tekrarlar farklı adlı hardlink'lerle oluşturulur (Ultralytics aynı dosya yolunu tekilleştirdiği için).
Nadir sınıf içeren görüntüler çoğunlukla testin %44'ünü oluşturan 1400×788 grubundan geldiği için bu yöntem
eğitim dağılımını teste de yaklaştırır.

In [ ]:
cls_of = {}
for i in train_ids:
    txt = (DS / "labels/train" / f"{i}.txt").read_text().splitlines()
    cls_of[i] = {int(s.split()[0]) for s in txt if s.strip()}

f_c = [sum(c in s for s in cls_of.values()) / len(train_ids) for c in range(4)]
r_c = [max(1.0, math.sqrt(RFS_T / f)) if f > 0 else 1.0 for f in f_c]

rng = random.Random(SEED)
lines, extra = [], 0
for i in train_ids:
    r = max((r_c[c] for c in cls_of[i]), default=1.0)
    k = int(r) + (1 if rng.random() < r - int(r) else 0)
    lines.append(f"./images/train/{i}.jpg")
    for j in range(1, k):
        name = f"{i}__rfs{j}"
        for sub, ext in (("images/train", ".jpg"), ("labels/train", ".txt")):
            dst = DS / sub / f"{name}{ext}"
            if not dst.exists():
                os.link(DS / sub / f"{i}{ext}", dst)
        lines.append(f"./images/train/{name}.jpg")
        extra += 1
(DS / "train_rfs.txt").write_text("\n".join(lines) + "\n")

data_yaml = WORK / "data_v2.yaml"
data_yaml.write_text(f"path: {DS}\ntrain: train_rfs.txt\nval: val.txt\nnames:\n"
                     + "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES)))

log(f"RFS: +{extra} görüntü tekrarı ({extra / len(train_ids):.1%}) → eğitim listesi {len(lines)} görüntü")
pd.DataFrame({"görüntü frekansı f_c": f_c, "tekrar faktörü r_c": r_c}, index=CLASSES).round(3)

<a id="4"></a>
## 4 · LR'ye bağlı azalan veri artırımı

Ultralytics yalnızca son `close_mosaic` epoch'ta mosaic'i kapatır. Burada bu fikir sürekli hâle getirilir:
her epoch başında veri artırımı gücü, cosine öğrenme oranı takvimiyle orantılı olarak azaltılır.

$$ \text{faktör}(e) = F + (1 - F)\cdot\frac{\text{lf}(e) - \text{lrf}}{1 - \text{lrf}}, \qquad F = 0.3 $$

Etkilenen ayarlar: mosaic olasılığı, `scale`, `translate`, `hsv_h`, `hsv_s`, `hsv_v`. `fliplr` sabit kalır.
Son `CLOSE_N` epoch'ta mosaic tamamen kapanır.

**Neden dosyaya yazılıyor?** 2× T4 eğitiminde Ultralytics her GPU için ayrı bir süreç başlatır ve özel trainer'ı
bu süreçlerde yeniden kurar. Sınıfın import edilebilir bir modülde olması bunu güvenilir kılar. Her epoch'un
değerleri `aug_log.jsonl` dosyasına kaydedilir.

In [ ]:
%%writefile aug_decay.py
"""LR schedule-coupled augmentation decay for Ultralytics detection training (DDP-safe)."""
import json
import os
from copy import copy

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils import DEFAULT_CFG, LOGGER, RANK

KEYS = ("mosaic", "scale", "translate", "hsv_h", "hsv_s", "hsv_v")


def on_epoch_start(tr):
    """Scale augmentation strength with the LR factor; close mosaic in the last `_close_n` epochs."""
    floor = float(os.environ.get("AUG_FLOOR", 0.3))
    if not hasattr(tr, "_aug_base"):
        tr._aug_base = {k: float(getattr(tr.args, k)) for k in KEYS}
    e, E, lrf = tr.epoch, tr.epochs, float(tr.args.lrf)
    lf = float(tr.lf(e)) if tr.lf is not None else 1.0
    fac = floor + (1 - floor) * min(max((lf - lrf) / (1 - lrf), 0.0), 1.0)
    closed = e >= E - tr._close_n
    hyp = copy(tr.args)
    for k, v in tr._aug_base.items():
        setattr(hyp, k, v * fac)
    ds = tr.train_loader.dataset
    if closed:
        hyp.mosaic = hyp.copy_paste = hyp.mixup = hyp.cutmix = 0.0
        if hasattr(ds, "mosaic"):
            ds.mosaic = False
    ds.transforms = ds.build_transforms(hyp=hyp)
    tr.train_loader.reset()                      # workers pick up the new transforms
    tr._aug_log = dict(epoch=e + 1, epochs=E, lr_factor=round(lf, 4), factor=round(fac, 3),
                       mosaic=round(hyp.mosaic, 3), scale=round(hyp.scale, 3), translate=round(hyp.translate, 3),
                       hsv_h=round(hyp.hsv_h, 4), hsv_s=round(hyp.hsv_s, 3), hsv_v=round(hyp.hsv_v, 3), closed=closed)
    if RANK in (-1, 0):
        LOGGER.info(f"[aug-decay] {tr._aug_log}")
        with open(os.path.join(str(tr.save_dir), "aug_log.jsonl"), "a") as f:
            f.write(json.dumps(tr._aug_log) + "\n")


class AugDecayTrainer(DetectionTrainer):
    def __init__(self, cfg=DEFAULT_CFG, overrides=None, _callbacks=None):
        super().__init__(cfg, overrides, _callbacks)
        # read from the environment so DDP subprocesses (which receive the already-zeroed args) agree
        self._close_n = int(os.environ["AUG_CLOSE_N"])
        self.args.close_mosaic = 0               # mosaic closing is handled in on_epoch_start
        if not any(getattr(c, "__name__", "") == "on_epoch_start" for c in self.callbacks["on_train_epoch_start"]):
            self.add_callback("on_train_epoch_start", on_epoch_start)

In [ ]:
os.environ["AUG_CLOSE_N"] = str(CLOSE_N)
os.environ["AUG_FLOOR"] = str(AUG_FLOOR)
os.environ["PYTHONPATH"] = f"{Path.cwd()}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"
sys.path.insert(0, str(Path.cwd()))

from aug_decay import AugDecayTrainer
from ultralytics import YOLO

# Takvimin önizlemesi: epoch sayısı TIME_H'ye göre belirlenir, burada 40 epoch için gösterilir
E, lrf = 40, 0.01
lf = lambda x: ((1 - math.cos(x * math.pi / E)) / 2) * (lrf - 1) + 1
fac = [AUG_FLOOR + (1 - AUG_FLOOR) * (lf(e) - lrf) / (1 - lrf) for e in range(E)]
mos = [f if e < E - CLOSE_N else 0 for e, f in enumerate(fac)]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, E + 1), [lf(e) for e in range(E)], label="LR faktörü", lw=2)
ax.plot(range(1, E + 1), fac, label="veri artırımı faktörü", lw=2)
ax.plot(range(1, E + 1), mos, label="mosaic olasılığı", lw=2, ls="--")
ax.set_xlabel("epoch"); ax.set_title("Örnek takvim (40 epoch)"); ax.legend(frameon=False); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

<a id="5"></a>
## 5 · Eğitim

Eğitim `TIME_H` saatlik bütçeyle çalışır ve her epoch'ta v2 doğrulama kümesinde değerlendirilir.
Kayıtlar: `results.csv` (epoch metrikleri), `aug_log.jsonl`, grafikler ve her 5 epoch'ta bir checkpoint.

**Hata toleransı:** Özel trainer herhangi bir nedenle başlayamazsa, eğitim Ultralytics'in varsayılan trainer'ıyla
(`close_mosaic=10`) otomatik olarak yeniden başlatılır. En az bir epoch tamamlanmışsa, mevcut `last.pt` ile devam edilir.

In [ ]:
device = [int(d) for d in DEVICE.split(",")] if DEVICE not in ("cpu", "") else "cpu"
train_args = dict(
    data=str(data_yaml), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device=device,
    workers=0 if SMOKE else 4, cos_lr=True, close_mosaic=CLOSE_N, fliplr=0.5, flipud=0.0, degrees=0.0,
    seed=SEED, amp=not SMOKE, val=True, plots=True, save_period=5, patience=1000,
    project=str(WORK / "runs"), name=RUN_NAME, exist_ok=True,
)
pd.Series({k: v for k, v in train_args.items() if k not in ("project",)}, name="değer").to_frame()

In [ ]:
def find_run():
    """Ultralytics proje klasörlerini iç içe açabildiği için koşu klasörünü adıyla bulur."""
    hits = sorted(WORK.glob(f"**/{RUN_NAME}/weights/last.pt"), key=lambda p: p.stat().st_mtime)
    return hits[-1].parent.parent if hits else None


train_t0 = time.time()
for label, trainer_cls in [("azalan veri artırımı", AugDecayTrainer), ("varsayılan (close_mosaic)", None)]:
    if find_run() is not None:
        log("önceki denemeden last.pt mevcut: eğitim tekrarlanmıyor")
        break
    left_h = TIME_H - (time.time() - train_t0) / 3600
    kw = dict(train_args, time=None if SMOKE else round(left_h, 3))
    if trainer_cls is not None:
        kw["trainer"] = trainer_cls
    log(f"eğitim başlıyor [{label}], bütçe {kw['time']} saat")
    try:
        YOLO(MODEL).train(**kw)
        break
    except Exception as e:
        log(f"eğitim [{label}] hata verdi: {type(e).__name__}: {e}")

RUN = find_run()
assert RUN is not None, "ağırlık üretilmedi"
log(f"eğitim süresi {(time.time() - train_t0) / 3600:.2f} saat | koşu klasörü: {RUN}")

<a id="6"></a>
## 6 · Eğitim geçmişi

`history.csv`: Ultralytics'in epoch kayıtları + epoch süresi + o epoch'taki veri artırımı değerleri.
Buradaki mAP değerleri Ultralytics'in kendi hesabıdır (yarışma metriğinden birkaç puan yüksek okur);
yarışma metriğiyle değerlendirme Bölüm 7'dedir.

In [ ]:
h = pd.read_csv(RUN / "results.csv")
h.columns = h.columns.str.strip()
h["epoch_time_s"] = h["time"].diff().fillna(h["time"])
aug_path = RUN / "aug_log.jsonl"
if aug_path.exists():
    aug = pd.DataFrame([json.loads(s) for s in aug_path.read_text().splitlines() if s.strip()])
    aug = aug.drop_duplicates("epoch", keep="last").drop(columns=["epochs"], errors="ignore")
    h = h.merge(aug.add_prefix("aug_").rename(columns={"aug_epoch": "epoch"}), on="epoch", how="left")
h.to_csv(WORK / "history.csv", index=False)

print(f"{len(h)} epoch · ortalama {h.epoch_time_s.mean():.0f} sn/epoch · toplam {h.epoch_time_s.sum() / 3600:.2f} saat")
show = [c for c in ["epoch", "epoch_time_s", "train/box_loss", "train/cls_loss", "val/box_loss", "val/cls_loss",
                    "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "lr/pg0", "aug_factor",
                    "aug_mosaic", "aug_closed"] if c in h.columns]
h[show]

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for c, lbl in [("train/box_loss", "train box"), ("val/box_loss", "val box"),
               ("train/cls_loss", "train cls"), ("val/cls_loss", "val cls")]:
    if c in h: axes[0].plot(h.epoch, h[c], label=lbl)
axes[0].set_title("Kayıplar"); axes[0].legend(frameon=False, fontsize=8)
for c, lbl in [("metrics/mAP50(B)", "mAP50"), ("metrics/precision(B)", "precision"), ("metrics/recall(B)", "recall")]:
    if c in h: axes[1].plot(h.epoch, h[c], label=lbl)
axes[1].set_title("Doğrulama metrikleri (Ultralytics)"); axes[1].legend(frameon=False, fontsize=8)
if "lr/pg0" in h: axes[2].plot(h.epoch, h["lr/pg0"] / h["lr/pg0"].max(), label="LR (normalize)")
if "aug_factor" in h: axes[2].plot(h.epoch, h["aug_factor"], label="veri artırımı faktörü")
if "aug_mosaic" in h: axes[2].plot(h.epoch, h["aug_mosaic"], ls="--", label="mosaic olasılığı")
axes[2].set_title("LR ve veri artırımı takvimi"); axes[2].legend(frameon=False, fontsize=8)
axes[3].bar(h.epoch, h.epoch_time_s / 60); axes[3].set_title("Epoch süresi (dk)")
for ax in axes: ax.set_xlabel("epoch"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

<a id="7"></a>
## 7 · Değerlendirme

### 7.1 Yarışma metriği
Kaggle kuralının birebir uygulaması: sınıf bazında güvene göre sıralama, IoU ≥ 0,5 ile açgözlü eşleştirme,
eşleşmemiş 200 px² altı tahminler yok sayılır, all-point interpolasyonlu AP, dört sınıfın ortalaması.
İsteğe bağlı görüntü ağırlıklarını destekler.

### 7.2 Test ağırlıklı doğrulama
v2 doğrulama kümesi test dağılımını olabildiğince taklit eder; ancak etiketli veride yalnızca 224 karanlık 1400×788
görüntü bulunduğundan tam eşleşme mümkün değildir. Bu yüzden her doğrulama görüntüsüne
`w = p_test(katman) / p_val(katman)` ağırlığı verilir (katman = çözünürlük × karanlık/aydınlık).
**Model seçimi test ağırlıklı mAP'e göre yapılır.**

In [ ]:
def image_info(folder, ids):
    rows = []
    for i in ids:
        with Image.open(DS / folder / f"{i}.jpg") as im:
            w, h_ = im.size
            im.draft("L", (128, 128))
            b = float(np.asarray(im.convert("L").resize((64, 64))).mean())
        rows.append((i, w, h_, f"{w}x{h_}" + (" · karanlık" if b < DARK_THR else " · aydınlık")))
    return pd.DataFrame(rows, columns=["image_id", "W", "H", "katman"]).set_index("image_id")


val_info, test_info = image_info("images/train", val_ids), image_info("images/test", test_ids)
p_test, p_val = test_info.katman.value_counts(normalize=True), val_info.katman.value_counts(normalize=True)
w_img = val_info.katman.map((p_test / p_val).fillna(0.0)).fillna(0.0)
w_img = w_img * len(w_img) / w_img.sum()
log(f"etkin örnek sayısı (ESS): {w_img.sum() ** 2 / (w_img ** 2).sum():.0f} / {len(w_img)}")

strata = pd.DataFrame({"val %": p_val * 100, "test %": p_test * 100,
                       "ağırlıklı val %": w_img.groupby(val_info.katman).sum() / w_img.sum() * 100}).fillna(0)
strata.sort_values("test %", ascending=False).round(2)

In [ ]:
# Doğrulama kümesinin gerçek kutuları (piksel, sol üst köşe)
gt_rows = []
for i in val_ids:
    W, H = val_info.loc[i, ["W", "H"]]
    for s in (DS / "labels/train" / f"{i}.txt").read_text().splitlines():
        if s.strip():
            c, cx, cy, bw, bh = s.split()
            cx, cy, bw, bh = float(cx) * W, float(cy) * H, float(bw) * W, float(bh) * H
            gt_rows.append((i, CLASSES[int(c)], cx - bw / 2, cy - bh / 2, bw, bh))
gt = pd.DataFrame(gt_rows, columns=["image_id", "label", "x", "y", "w", "h"])


def _iou(b, B):
    x1 = np.maximum(b[0], B[:, 0]); y1 = np.maximum(b[1], B[:, 1])
    x2 = np.minimum(b[0] + b[2], B[:, 0] + B[:, 2]); y2 = np.minimum(b[1] + b[3], B[:, 1] + B[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    return inter / np.maximum(b[2] * b[3] + B[:, 2] * B[:, 3] - inter, 1e-9)


def evaluate(pred, ids, weights=None, min_area=200.0):
    """Yarışma tarzı mAP@0.5 (isteğe bağlı görüntü ağırlıklarıyla)."""
    ids = list(ids)
    w = pd.Series(1.0, index=ids) if weights is None else weights.reindex(ids).fillna(0.0)
    keep = set(w.index[w > 0])
    g_all, p_all = gt[gt.image_id.isin(keep)], pred[pred.image_id.isin(keep)]
    ap = {}
    for c in CLASSES:
        g = g_all[g_all.label == c]
        p = p_all[p_all.label == c].sort_values("conf", ascending=False, kind="mergesort")
        boxes = {k: v[["x", "y", "w", "h"]].to_numpy(float) for k, v in g.groupby("image_id")}
        used = {k: np.zeros(len(v), bool) for k, v in boxes.items()}
        npos = float(w.reindex(g.image_id).sum())
        if npos == 0:
            ap[c] = float("nan"); continue
        tp, fp = np.zeros(len(p)), np.zeros(len(p))
        pw = w.reindex(p.image_id).to_numpy()
        for k, (img, b) in enumerate(zip(p.image_id.to_numpy(), p[["x", "y", "w", "h"]].to_numpy(float))):
            B, j, best = boxes.get(img), -1, 0.0
            if B is not None:
                ious = _iou(b, B); ious[used[img]] = -1.0
                j = int(np.argmax(ious)); best = ious[j]
            if best >= 0.5:
                used[img][j] = True; tp[k] = pw[k]
            elif b[2] * b[3] >= min_area:
                fp[k] = pw[k]
        ctp, cfp = np.cumsum(tp), np.cumsum(fp)
        rec = np.concatenate([[0.0], ctp / npos, [1.0]])
        prec = np.concatenate([[0.0], ctp / np.maximum(ctp + cfp, 1e-12), [0.0]])
        prec = np.maximum.accumulate(prec[::-1])[::-1]
        idx = np.where(rec[1:] != rec[:-1])[0]
        ap[c] = float(np.sum((rec[idx + 1] - rec[idx]) * prec[idx + 1]))
    vals = [v for v in ap.values() if not np.isnan(v)]
    return {"mAP": float(np.mean(vals)) if vals else float("nan"), **ap}


def predict(weights, folder, ids):
    """Görüntü görüntü tahmin (liste kaynağı Ultralytics'te tek batch sayıldığı için bellek taşar)."""
    m = YOLO(str(weights))
    rows = []
    for i in ids:
        r = m.predict(str(DS / folder / f"{i}.jpg"), imgsz=IMGSZ, conf=0.001, iou=0.7, max_det=1000,
                      device=0 if torch.cuda.is_available() else "cpu", half=torch.cuda.is_available(),
                      verbose=False)[0]
        for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            if x2 - x1 < 1 or y2 - y1 < 1:      # görüntü kenarında sıfır boyuta kırpılmış kutu: geçersiz
                continue
            rows.append((i, CLASSES[int(c)], round(s, 5), round(x1, 1), round(y1, 1),
                         round(x2 - x1, 1), round(y2 - y1, 1)))
    return pd.DataFrame(rows, columns=["image_id", "label", "conf", "x", "y", "w", "h"])

In [ ]:
subsets = {n: read_ids(f"{n}.txt") for n in ["val_1400x788", "val_dark", "val_1400x788_dark"] if (DS / f"{n}.txt").exists()}
val_preds, scores = {}, {}
for tag in ["last", "best"]:
    wpath = RUN / "weights" / f"{tag}.pt"
    if not wpath.exists():
        continue
    pv = predict(wpath, "images/train", val_ids)
    pv.to_csv(WORK / f"preds_val_{tag}.csv", index=False)
    val_preds[tag] = pv
    sc = {"val (ağırlıksız)": evaluate(pv, val_ids), "val (test ağırlıklı)": evaluate(pv, val_ids, w_img)}
    for n, ids in subsets.items():
        sc[n] = evaluate(pv, ids)
    scores[tag] = sc
    log(f"{tag}.pt: test ağırlıklı mAP@0.5 = {sc['val (test ağırlıklı)']['mAP']:.4f}")

chosen = max(scores, key=lambda t: scores[t]["val (test ağırlıklı)"]["mAP"])
table = pd.DataFrame({(t, k): v for t, sc in scores.items() for k, v in sc.items()}).T
table.index.names = ["ağırlık", "küme"]
table

In [ ]:
for name in ["confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    if (RUN / name).exists():
        fig = plt.figure(figsize=(8, 6)); plt.imshow(plt.imread(RUN / name)); plt.axis("off"); plt.title(name); plt.show()

<a id="8"></a>
## 8 · Test tahmini ve submission

Seçilen ağırlıkla (test ağırlıklı doğrulama mAP'i en yüksek olan) test görüntüleri tahmin edilir.
Dosya yarışma formatına göre doğrulanır: her test görüntüsü için bir satır, tahmin yoksa `none`,
`label conf x y w h` grupları, `0 ≤ conf ≤ 1`, `w, h > 0`. Her iki ağırlığın submission'ı da saklanır.

In [ ]:
def write_submission(test_pred, path):
    s = (test_pred.label + " " + test_pred.conf.astype(str) + " " + test_pred.x.astype(str) + " "
         + test_pred.y.astype(str) + " " + test_pred.w.astype(str) + " " + test_pred.h.astype(str))
    strings = s.groupby(test_pred.image_id).agg(" ".join)
    ids = sorted(test_ids)
    sub = pd.DataFrame({"image_id": ids, "PredictionString": [strings.get(i, "none") for i in ids]})
    assert len(sub) == len(set(test_ids)) and sub.PredictionString.str.len().gt(0).all()
    for ps in sub.PredictionString:
        if ps != "none":
            t = ps.split()
            assert len(t) % 6 == 0
            for k in range(0, len(t), 6):
                assert t[k] in CLASSES and 0 <= float(t[k + 1]) <= 1 and float(t[k + 4]) > 0 and float(t[k + 5]) > 0
    sub.to_csv(path, index=False)
    return sub


for tag in scores:
    pt = predict(RUN / "weights" / f"{tag}.pt", "images/test", test_ids)
    pt.to_csv(WORK / f"preds_test_{tag}.csv", index=False)
    sub = write_submission(pt, WORK / f"submission_{tag}.csv")
    log(f"submission_{tag}.csv: {len(sub)} satır, {int((sub.PredictionString == 'none').sum())} boş, {len(pt)} kutu")

shutil.copy(WORK / f"submission_{chosen}.csv", WORK / "submission.csv")
json.dump({"chosen": chosen, "scores": scores, "rfs": {"t": RFS_T, "f_c": f_c, "r_c": r_c, "extra_images": extra},
           "imgsz": IMGSZ, "time_h": TIME_H, "ultralytics": ultralytics.__version__},
          open(WORK / "scores.json", "w"), indent=2)
log(f"submission.csv ← {chosen}.pt")
pd.read_csv(WORK / "submission.csv").head()

<a id="9"></a>
## 9 · Özet

In [ ]:
best = scores[chosen]
summary = pd.Series({
    "seçilen ağırlık": f"{chosen}.pt",
    "test ağırlıklı val mAP@0.5": round(best["val (test ağırlıklı)"]["mAP"], 4),
    "ağırlıksız val mAP@0.5": round(best["val (ağırlıksız)"]["mAP"], 4),
    **{f"AP {c} (test ağırlıklı)": round(best["val (test ağırlıklı)"][c], 4) for c in CLASSES},
    "epoch": len(h),
    "ortalama epoch süresi (sn)": round(h.epoch_time_s.mean()),
    "RFS ek görüntü": extra,
    "toplam süre (saat)": round((time.time() - T0) / 3600, 2),
}, name="değer")
if not SMOKE:
    shutil.rmtree(DS, ignore_errors=True)       # çıktıyı küçük tut: yalnızca ağırlıklar, csv ve json kalır
summary.to_frame()

**Çıktılar** (`/kaggle/working`)

| Dosya | İçerik |
|---|---|
| `submission.csv` | Seçilen ağırlıkla yarışma formatında test tahminleri |
| `submission_last.csv`, `submission_best.csv` | İki ağırlığın submission'ları |
| `scores.json` | Yarışma metriğiyle tüm skorlar, RFS parametreleri |
| `history.csv` | Epoch bazında metrikler, süre ve veri artırımı değerleri |
| `preds_val_*.csv`, `preds_test_*.csv` | Ham tahminler (`image_id,label,conf,x,y,w,h`) |
| `runs/y11s_v2/` | `last.pt`, `best.pt`, 5 epoch'luk checkpoint'ler, eğitim grafikleri |

**Sonraki adımlar:** tile'lı (SAHI) tahmin, 1536 çözünürlükte tahmin, YOLO11m ile aynı tarif, WBF ensemble.